In [50]:
import pandas as pd

## Создаем датафрейм с двумя столбцами

In [51]:
views = pd.read_csv(
    '../data/feed-views.log',
    sep='\t',
    names=['datetime', 'user']
)

In [52]:
print("\nТип данных ПОСЛЕ преобразования:")
print(views['datetime'].dtype)
print("\nПример значения:")
print(views['datetime'].iloc[0])


Тип данных ПОСЛЕ преобразования:
str

Пример значения:
2020-04-17 12:01:08.463179


## Приводим столбец datetime к типу datetime64[ns]

In [53]:
views['datetime'] = pd.to_datetime(views['datetime'])

In [54]:
print("\nТип данных ПОСЛЕ преобразования:")
print(views['datetime'].dtype)
print("\nПример значения:")
print(views['datetime'].iloc[0])


Тип данных ПОСЛЕ преобразования:
datetime64[us]

Пример значения:
2020-04-17 12:01:08.463179


## Извлечение компонентов даты и времени

In [55]:
views['year'] = views['datetime'].dt.year
views['month'] = views['datetime'].dt.month
views['day'] = views['datetime'].dt.day
views['hour'] = views['datetime'].dt.hour
views['minute'] = views['datetime'].dt.minute
views['second'] = views['datetime'].dt.second

## Создание столбца daytime с использованием cut

In [56]:
bins = [-1, 4, 7, 11, 17, 20, 24]  # -1 для включения 0
labels = ['night', 'early morning', 'morning', 'afternoon', 'early evening', 'evening']

In [57]:
views['daytime']=pd.cut(views['hour'], bins=bins, labels=labels, right=False)
print(views['daytime'].value_counts().sort_index())

daytime
night            129
early morning      5
morning           36
afternoon        252
early evening    145
evening          509
Name: count, dtype: int64


## Установка user как индекса

In [58]:
views.set_index('user', inplace=True)

## Подсчет количества элементов

In [59]:
views.count()


datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [60]:
views['daytime'].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

## Сортировка по часам, минутам и секундам

In [61]:
views.sort_values(['hour', 'minute', 'second'])

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night
...,...,...,...,...,...,...,...,...
artem,2020-05-21 23:49:22.386789,2020,5,21,23,49,22,evening
anatoliy,2020-05-09 23:53:55.599821,2020,5,9,23,53,55,evening
pavel,2020-05-09 23:54:54.260791,2020,5,9,23,54,54,evening


## Минимум, максимум и мода

In [62]:
print(f"Минимальный час: {views['hour'].min()}")
print(f"Максимальный час: {views['hour'].max()}")

Минимальный час: 0
Максимальный час: 23


In [63]:
daytime_mode = views['daytime'].mode()
print(f"\nМода для daytime: {daytime_mode.iloc[0]}")


Мода для daytime: evening


## Максимальный час для строк, где время суток - night

In [64]:
night_mask = views['daytime'] == 'night'
max_hour_night = views[night_mask]['hour'].max()
print(f'максимальный час: {max_hour_night}')
night_users = views[night_mask & (views['hour']==max_hour_night)].index.unique()
print(f"Пользователь, посещавший {max_hour_night} часов (night): {list(night_users)}")

views.loc[views.daytime == 'night'].hour.idxmax()

максимальный час: 3
Пользователь, посещавший 3 часов (night): ['konstantin']


'konstantin'

## Минимальный час для строк, где время суток - morning

In [65]:
morning_mask = views['daytime'] == 'morning'
min_hour_morning = views[morning_mask]['hour'].min()
print(f'минимальный час: {min_hour_morning}')
morning_users = views[morning_mask & (views['hour'] == min_hour_morning)].index.unique()
print(f"Пользователь, посещавший {min_hour_morning} часов (morning): {list(morning_users)}")
views.loc[views.daytime == 'morning'].hour.idxmin() 

минимальный час: 8
Пользователь, посещавший 8 часов (morning): ['alexander']


'alexander'

In [66]:
hour_mode = views['hour'].mode()
print(f"Мода для часа (наиболее частые часы): {list(hour_mode)}")

Мода для часа (наиболее частые часы): [22]


## Три самых ранних и три самых поздних часа

In [67]:
earliest_hours = views.nsmallest(3, 'hour')
print("Три самых ранних часа и соответствующие пользователи:")
for idx, row in earliest_hours.iterrows():
    print(f"  Час {row['hour']}:{row['minute']:02d}:{row['second']:02d} - пользователь {idx}")


latest_hours = views.nlargest(3, 'hour')
print("\nТри самых поздних часа и соответствующие пользователи:")
for idx, row in latest_hours.iterrows():
    print(f"  Час {row['hour']}:{row['minute']:02d}:{row['second']:02d} - пользователь {idx}")

Три самых ранних часа и соответствующие пользователи:
  Час 0:30:45 - пользователь artem
  Час 0:17:22 - пользователь konstantin
  Час 0:17:28 - пользователь konstantin

Три самых поздних часа и соответствующие пользователи:
  Час 23:06:34 - пользователь konstantin
  Час 23:40:32 - пользователь artem
  Час 23:10:03 - пользователь artem


## Базовая статистика и интерквартильный размах

In [68]:
stats = views.describe()
print(stats)

                         datetime    year        month          day  \
count                        1076  1076.0  1076.000000  1076.000000   
mean   2020-05-10 09:00:41.211420  2020.0     4.870818    13.552974   
min    2020-04-17 12:01:08.463179  2020.0     4.000000     1.000000   
25%    2020-05-10 01:13:49.857472  2020.0     5.000000    11.000000   
50%    2020-05-11 22:48:35.302553  2020.0     5.000000    13.000000   
75%    2020-05-14 14:44:34.749530  2020.0     5.000000    15.000000   
max    2020-05-22 10:36:14.662600  2020.0     5.000000    30.000000   
std                           NaN     0.0     0.335557     4.906567   

              hour       minute       second  
count  1076.000000  1076.000000  1076.000000  
mean     16.249071    29.629182    29.500929  
min       0.000000     0.000000     0.000000  
25%      13.000000    14.000000    14.000000  
50%      19.000000    29.000000    30.000000  
75%      22.000000    46.000000    45.000000  
max      23.000000    59.000000

In [69]:
Q1 = stats.loc['25%', 'hour']
Q3 = stats.loc['75%', 'hour']
iqr = Q3 - Q1

print(f"Интерквартильный размах (IQR) для часа: {iqr}")

Интерквартильный размах (IQR) для часа: 9.0


In [70]:
views.info()

<class 'pandas.DataFrame'>
Index: 1076 entries, artem to artem
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  1076 non-null   datetime64[us]
 1   year      1076 non-null   int32         
 2   month     1076 non-null   int32         
 3   day       1076 non-null   int32         
 4   hour      1076 non-null   int32         
 5   minute    1076 non-null   int32         
 6   second    1076 non-null   int32         
 7   daytime   1076 non-null   category      
dtypes: category(1), datetime64[us](1), int32(6)
memory usage: 75.6+ KB
